In [1]:
from __future__ import annotations

import importlib
import re
from dataclasses import dataclass
from typing import Optional

import pandas as pd

import entity_binding
import interjection_filter
import speech_grouping
import title_cleaning

importlib.reload(entity_binding)
importlib.reload(interjection_filter)
importlib.reload(speech_grouping)
importlib.reload(title_cleaning)

# from entity_binding import find_id_mp
from entity_binding import bind_entities, bind_report
from interjection_filter import filter_reaction_interjections
from speech_grouping import group_consecutive_speeches
from title_cleaning import clean_title


In [166]:
df = pd.read_csv('../data/speech/1996_2019/parisparl_reden_1998.csv')

df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Jour test pour construire les regles de structure.
one_day_speech = df[df['date'].eq(pd.Timestamp('1998-11-25'))].copy()
one_day_speech['row_order'] = range(1, len(one_day_speech) + 1)
one_day_speech


,id,speaker,date,party,parliamentary_group,role,lp,session,year,agenda_item,agenda_item_type,interjection,text,row_order
78080,122427,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,1.0,NaN,False,"\"""" La séance est ouverte. ( La séance est ouv...",1
78081,122428,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,1.0,NaN,False,"\"""" 1 SOUHAITS DE BIENVENUE à les PRÉSIDENTS D...",2
78082,122430,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,1.0,NaN,False,"\"""" Mes chers collègues, sur décision de notre...",3
78083,122431,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,1.0,NaN,True,"\"""" ( Mmes et MM. les députés, ainsi que Mmes ...",4
78084,122432,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,1.0,NaN,False,"\"""" 2 QUESTIONS à le GOUVERNEMENT \""""",5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80628,126655,le président,1998-11-25,NaN,NaN,presidency,11,99.0,1998,3.0,NaN,True,"\"""". \""""",2549
80629,126656,le président,1998-11-25,NaN,NaN,presidency,11,99.0,1998,3.0,NaN,False,"\"""" A quinze heures, deuxième séance publique:...",2550
80630,126657,le président,1998-11-25,NaN,NaN,presidency,11,99.0,1998,3.0,NaN,False,"\"""" Suite de l' ordre de le jour de la premièr...",2551
80631,126658,le président,1998-11-25,NaN,NaN,presidency,11,99.0,1998,3.0,NaN,False,"\"""" Suite de l' ordre de le jour de la premièr...",2552


In [167]:
# Vue utile pour inspecter les lignes marquees comme interjections,
# sans perdre les discours qui doivent heriter du sujet/sous-sujet courant.
one_day_interjections = filter_reaction_interjections(one_day_speech)
one_day_interjections


,id,speaker,date,party,parliamentary_group,role,lp,session,year,agenda_item,agenda_item_type,interjection,text,row_order
78083,122431,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,1.0,NaN,True,"\"""" ( Mmes et MM. les députés, ainsi que Mmes ...",4
78092,122443,Valéry Giscard d’Estaing,1998-11-25,NaN,UDF,mp,11,98.0,1998,2.0,NaN,True,"\"""" ( Sourires. ) \""""",13
78095,122446,Valéry Giscard d’Estaing,1998-11-25,NaN,UDF,mp,11,98.0,1998,2.0,NaN,True,"\"""" ( Rires et applaudissements sur de nombreu...",16
78097,122449,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,2.0,NaN,True,"\"""" ( Sourires. ) \""""",18
78100,122453,Valéry Giscard d’Estaing,1998-11-25,NaN,UDF,mp,11,98.0,1998,2.0,NaN,True,"\"""" ( Sourires. ) \""""",21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80408,126303,le président,1998-11-25,NaN,NaN,presidency,11,99.0,1998,0.0,NaN,True,"\"""" ( Sourires. ) \""""",2329
80460,126390,NaN,1998-11-25,NaN,NaN,gov,11,99.0,1998,0.0,NaN,True,"\"""" ( Applaudissements sur les bancs de le gro...",2381
80470,126408,Nicole Catala,1998-11-25,NaN,RPR,mp,11,99.0,1998,0.0,NaN,True,"\"""" ( Exclamations sur les bancs de le groupe ...",2391
80472,126410,Nicole Catala,1998-11-25,NaN,RPR,mp,11,99.0,1998,0.0,NaN,True,"\"""" ( Mêmes mouvements. ) \""""",2393


In [168]:
# Le groupement final est construit après annotate_debate_structure,
# pour isoler major_topic et subtopic en lignes procedure.


## Preprocessing de la structure du debat

Objectif: recuperer proprement la structure d'une journee de debat a partir de `one_day_speech`.

Regles de depart:

- un **grand sujet** commence par un numero suivi d'un titre majoritairement en capitales, par exemple `1 SOUHAITS DE BIENVENUE...`;
- un **sous-sujet** est une ligne courte, sans numero, majoritairement en capitales, par exemple `TAUX DE CONVERSION DE L' EURO`;
- les lignes suivantes heritent du dernier grand sujet et du dernier sous-sujet detectes.


In [169]:
MAJOR_TOPIC_RE = re.compile(r"^\s*(?P<number>\d{1,2})\s+(?P<title>.+?)\s*$")
INLINE_MAJOR_TOPIC_RE = re.compile(r"(?:^|[.!?]\s+)(?P<number>\d{1,2})\s+(?P<title>.+?)\s*$")
TRAILING_MAJOR_DETAIL_RE = re.compile(
    r"\s+(?:Explications de vote|Discussion générale|Suite de la discussion|Discussion d['’ ]un|Vote sur).*$"
)
WRAPPER_RE = re.compile(r"""^[\s\\\"'“”«»]+|[\s\\\"'“”«»]+$""")
WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9']+")
INLINE_TOPIC_RE = re.compile(
    r"\b(?:Nous\s+(?:passons|en\s+revenons|revenons)|On\s+en\s+revient)\b.+?\s+"
    r"(?P<title>[A-ZÀ-ÖØ-Þ][A-ZÀ-ÖØ-Þ' -]+[A-ZÀ-ÖØ-Þ])\s*$",
)


@dataclass(frozen=True)
class TopicMatch:
    number: Optional[int]
    title: str


def clean_structure_text(value: object) -> str:
    """Nettoie les guillemets parasites sans modifier les mots du debat."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    text = WRAPPER_RE.sub("", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def uppercase_ratio(text: str) -> float:
    letters = [char for char in text if char.isalpha()]
    if not letters:
        return 0.0
    return sum(char.isupper() for char in letters) / len(letters)


def is_parenthetical(text: str) -> bool:
    stripped = text.strip()
    return stripped.startswith("(") and stripped.endswith(")")


def is_title_like(text: str, min_ratio: float = 0.70, max_chars: int = 180) -> bool:
    text = clean_structure_text(text)
    if not text or is_parenthetical(text):
        return False
    if len(text) > max_chars:
        return False
    if text.endswith((".", "!", "?", ";", ":")):
        return False
    return uppercase_ratio(text) >= min_ratio


def normalize_major_title(title: str) -> str:
    title = TRAILING_MAJOR_DETAIL_RE.sub("", title).strip()
    title = re.sub(r"\bMODIFICATIONDE\b", "MODIFICATION DE", title)
    title = re.sub(r"\bORDONNANCERELATIVE\b", "ORDONNANCE RELATIVE", title)
    title = re.sub(r"\s+", " ", title)
    return clean_title(title)


def match_major_topic(text: object) -> Optional[TopicMatch]:
    cleaned = clean_structure_text(text)
    match = MAJOR_TOPIC_RE.match(cleaned) or INLINE_MAJOR_TOPIC_RE.search(cleaned)
    if not match:
        return None

    title = normalize_major_title(match.group("title"))
    if not is_title_like(title, min_ratio=0.55):
        return None
    return TopicMatch(number=int(match.group("number")), title=title)


def match_subtopic_candidate(text: object) -> Optional[TopicMatch]:
    cleaned = clean_structure_text(text)
    if MAJOR_TOPIC_RE.match(cleaned):
        return None
    if not is_title_like(cleaned, min_ratio=0.75, max_chars=120):
        return None

    words = WORD_RE.findall(cleaned)
    if not 2 <= len(words) <= 14:
        return None
    return TopicMatch(number=None, title=clean_title(cleaned))


def extract_inline_subtopic_text(text: object) -> Optional[str]:
    cleaned = clean_structure_text(text)
    inline_match = INLINE_TOPIC_RE.search(cleaned)
    if inline_match:
        candidate = inline_match.group("title").strip()
        if match_subtopic_candidate(candidate) is not None:
            return candidate

    parts = re.split(r"[.!?;:]\s+", cleaned)
    if len(parts) < 2:
        return None

    candidate = parts[-1].strip()
    if re.search(r"\bla parole est\b", candidate, flags=re.IGNORECASE):
        return None
    if match_subtopic_candidate(candidate) is None:
        return None
    return candidate


def match_subtopic(text: object) -> Optional[TopicMatch]:
    direct_match = match_subtopic_candidate(text)
    if direct_match is not None:
        return direct_match

    inline_text = extract_inline_subtopic_text(text)
    if inline_text is None:
        return None
    return TopicMatch(number=None, title=clean_title(inline_text))


def is_major_topic_continuation(text: object) -> bool:
    cleaned = clean_structure_text(text)
    if not is_title_like(cleaned, min_ratio=0.70, max_chars=80):
        return False
    return cleaned.startswith(("DE ", "DU ", "DES ", "D'", "À ", "A "))

def annotate_debate_structure(one_day_speech: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    """Ajoute les colonnes de structure a un dataframe d'une journee de debat."""
    if text_col not in one_day_speech.columns:
        raise KeyError(f"Colonne texte absente: {text_col!r}")

    output = one_day_speech.copy()
    output["structure_text"] = output[text_col].map(clean_structure_text)
    output["structure_level"] = "speech"
    output["major_topic_number"] = pd.NA
    output["major_topic"] = pd.NA
    output["major_topic_order"] = pd.NA
    output["subtopic"] = pd.NA

    major_topic_order = 0
    current_major_number: Optional[int] = None
    current_major_title: Optional[str] = None
    current_major_order: Optional[int] = None
    current_subtopic: Optional[str] = None
    current_major_index = None
    awaiting_major_continuation = False

    for index, row in output.iterrows():
        text = row["structure_text"]

        major = match_major_topic(text)
        if major is not None:
            major_topic_order += 1
            current_major_number = major.number
            current_major_title = major.title
            current_major_order = major_topic_order
            current_subtopic = None
            current_major_index = index
            awaiting_major_continuation = True
            output.at[index, "structure_level"] = "major_topic"
            output.at[index, "major_topic_number"] = major.number
            output.at[index, "major_topic"] = major.title
            output.at[index, "major_topic_order"] = current_major_order
            continue

        if awaiting_major_continuation and is_major_topic_continuation(text):
            current_major_title = clean_title(f"{current_major_title} {text}")
            output.at[index, "structure_level"] = "speech"
            output.at[index, "major_topic_number"] = current_major_number
            output.at[index, "major_topic"] = current_major_title
            output.at[index, "major_topic_order"] = current_major_order
            if current_major_index is not None:
                output.at[current_major_index, "major_topic"] = current_major_title
            continue

        subtopic = match_subtopic(text)
        if subtopic is not None:
            current_subtopic = subtopic.title
            awaiting_major_continuation = False
            output.at[index, "structure_level"] = "subtopic"
        if subtopic is None:
            awaiting_major_continuation = False

        output.at[index, "major_topic_number"] = current_major_number
        output.at[index, "major_topic"] = current_major_title
        output.at[index, "major_topic_order"] = current_major_order
        output.at[index, "subtopic"] = current_subtopic

    return output


def extract_debate_structure(one_day_speech: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    annotated = annotate_debate_structure(one_day_speech, text_col=text_col)
    structure_rows = annotated["structure_level"].isin(["major_topic", "subtopic"])
    optional_context_cols = [col for col in ["session", "agenda_item"] if col in annotated.columns]
    structure_cols = [
        "id",
        *optional_context_cols,
        "speaker",
        "interjection",
        "structure_level",
        "major_topic_order",
        "major_topic_number",
        "major_topic",
        "subtopic",
        "structure_text",
    ]
    return annotated.loc[structure_rows, structure_cols]


In [170]:
annotated_one_day_speech = annotate_debate_structure(one_day_speech)
debate_structure = extract_debate_structure(one_day_speech)

major_topics_check = (
    debate_structure[debate_structure['structure_level'].isin(['major_topic', 'subtopic'])]
    [[col for col in ['structure_level', 'major_topic_order', 'major_topic_number', 'major_topic', 'subtopic', 'session', 'agenda_item', 'id', 'structure_text'] if col in debate_structure.columns]]
    .drop_duplicates(subset=['structure_level', 'major_topic_order', 'major_topic_number', 'major_topic', 'subtopic'])
    .sort_values(['major_topic_order', 'id'])
)
detected_numbers = set(major_topics_check['major_topic_number'].dropna().astype(int))
if detected_numbers:
    expected_numbers = set(range(min(detected_numbers), max(detected_numbers) + 1))
    print('Major topic numbers detected:', sorted(detected_numbers))
    print('Missing major topic numbers:', sorted(expected_numbers - detected_numbers))

print(debate_structure['structure_level'].value_counts())
major_topics_check


Major topic numbers detected: [1, 2, 3, 4, 5, 6, 7, 8]
Missing major topic numbers: []
structure_level
major_topic    14
subtopic       10
Name: count, dtype: int64


,structure_level,major_topic_order,major_topic_number,major_topic,subtopic,session,agenda_item,id,structure_text
78081,major_topic,1,1,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,<NA>,98.0,1.0,122428,1 SOUHAITS DE BIENVENUE à les PRÉSIDENTS D' AS...
78084,major_topic,2,2,QUESTIONS AU GOUVERNEMENT,<NA>,98.0,1.0,122432,2 QUESTIONS à le GOUVERNEMENT
78087,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,TAUX DE CONVERSION DE L'EURO,98.0,2.0,122436,TAUX DE CONVERSION DE L' EURO
78128,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,BILAN DES PRIVATISATIONS,98.0,2.0,122496,"Nous passons à le groupe Radical, Citoyen et V..."
78152,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,AIDE À DOMICILE,98.0,2.0,122531,AIDE À DOMICILE
78166,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,FISCALITÉ DES RETRAITÉS,98.0,2.0,122551,FISCALITÉ DES RETRAITÉS
78196,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,DÉPÉNALISATION DU CANNABIS,98.0,2.0,122594,DÉPÉNALISATION de le CANNABIS
78226,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,RETRAITE DES SALARIÉS AYANT COTISÉ QUARANTE AN...,98.0,2.0,122633,RETRAITE DES SALARIÉS AYANT COTISÉ QUARANTE AN...
78241,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,POLITIQUE AGRICOLE COMMUNE,98.0,2.0,122655,Nous passons à le groupe socialiste. POLITIQUE...
78268,subtopic,2,2,QUESTIONS AU GOUVERNEMENT,MAÎTRISE DES DÉPENSES DE SANTÉ,98.0,2.0,122695,MAÎTRISE DES DÉPENSES DE SANTÉ


In [171]:
grouped_one_day_speeches = group_consecutive_speeches(
    annotated_one_day_speech,
    source_order_col='row_order',
    procedure_col='structure_level',
)
grouped_one_day_speeches


,speech_order,row_type,procedure_type,first_row_order,last_row_order,source_row_orders,source_ids,speaker,role,party,parliamentary_group,date,session,agenda_item,major_topic_order,major_topic,subtopic,speech_text,interjections
0,1,speech,None,1,1,[1],[122427],le président,presidency,None,None,1998-11-25,98.0,1.0,NaN,None,None,La séance est ouverte. ( La séance est ouverte...,[]
1,2,procedure,major_topic,2,2,[2],[122428],le président,presidency,None,None,1998-11-25,98.0,1.0,1.0,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,None,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,[]
2,3,speech,None,3,4,"[3, 4]","[122430, 122431]",le président,presidency,None,None,1998-11-25,98.0,1.0,1.0,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,None,"Mes chers collègues, sur décision de notre bur...","[( Mmes et MM. les députés, ainsi que Mmes et ..."
3,4,procedure,major_topic,5,5,[5],[122432],le président,presidency,None,None,1998-11-25,98.0,1.0,2.0,QUESTIONS AU GOUVERNEMENT,None,QUESTIONS AU GOUVERNEMENT,[]
4,5,speech,None,6,7,"[6, 7]","[122434, 122435]",le président,presidency,None,None,1998-11-25,98.0,2.0,2.0,QUESTIONS AU GOUVERNEMENT,None,L' ordre de le jour appelle les questions à le...,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1659,1660,speech,None,2532,2539,"[2532, 2533, 2534, 2535, 2536, 2537, 2538, 2539]","[126636, 126637, 126638, 126639, 126640, 12664...",le président,presidency,None,None,1998-11-25,99.0,2.0,12.0,DÉPÔT DE RAPPORTS SUR DES PROPOSITIONS DE RÉSO...,None,"J' ai reçu, le 25 novembre 1998, de M. Yves Du...",[]
1660,1661,procedure,major_topic,2540,2540,[2540],[126644],le président,presidency,None,None,1998-11-25,99.0,2.0,13.0,COMMUNICATION RELATIVE AUX ASSEMBLÉES TERRITOR...,None,COMMUNICATION RELATIVE AUX ASSEMBLÉES TERRITOR...,[]
1661,1662,speech,None,2541,2543,"[2541, 2542, 2543]","[126646, 126647, 126648]",le président,presidency,None,None,1998-11-25,99.0,3.0,13.0,COMMUNICATION RELATIVE AUX ASSEMBLÉES TERRITOR...,None,J' ai reçu de M. le Premier ministre une lettr...,[]
1662,1663,procedure,major_topic,2544,2544,[2544],[126649],le président,presidency,None,None,1998-11-25,99.0,3.0,14.0,ORDRE DU JOUR DES PROCHAINES SÉANCES,None,ORDRE DU JOUR DES PROCHAINES SÉANCES,[]


In [172]:
final_structure = major_topics_check[['major_topic_order', 'major_topic', 'subtopic']]
final_structure

,major_topic_order,major_topic,subtopic
78081,1,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,<NA>
78084,2,QUESTIONS AU GOUVERNEMENT,<NA>
78087,2,QUESTIONS AU GOUVERNEMENT,TAUX DE CONVERSION DE L'EURO
78128,2,QUESTIONS AU GOUVERNEMENT,BILAN DES PRIVATISATIONS
78152,2,QUESTIONS AU GOUVERNEMENT,AIDE À DOMICILE
78166,2,QUESTIONS AU GOUVERNEMENT,FISCALITÉ DES RETRAITÉS
78196,2,QUESTIONS AU GOUVERNEMENT,DÉPÉNALISATION DU CANNABIS
78226,2,QUESTIONS AU GOUVERNEMENT,RETRAITE DES SALARIÉS AYANT COTISÉ QUARANTE AN...
78241,2,QUESTIONS AU GOUVERNEMENT,POLITIQUE AGRICOLE COMMUNE
78268,2,QUESTIONS AU GOUVERNEMENT,MAÎTRISE DES DÉPENSES DE SANTÉ


one_day_interjections = filter_reaction_interjections(one_day_speech)

Ici sont gardé ce qui sont censé être les interjections. Parfois pouvent comporté des faux positif. j'aimerai donc effectué un traitement rebosute pour gardé des vraies reactions comme appaludissement, sourire, murmures etc  

In [173]:
one_day_interjections

,id,speaker,date,party,parliamentary_group,role,lp,session,year,agenda_item,agenda_item_type,interjection,text,row_order
78083,122431,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,1.0,NaN,True,"\"""" ( Mmes et MM. les députés, ainsi que Mmes ...",4
78092,122443,Valéry Giscard d’Estaing,1998-11-25,NaN,UDF,mp,11,98.0,1998,2.0,NaN,True,"\"""" ( Sourires. ) \""""",13
78095,122446,Valéry Giscard d’Estaing,1998-11-25,NaN,UDF,mp,11,98.0,1998,2.0,NaN,True,"\"""" ( Rires et applaudissements sur de nombreu...",16
78097,122449,le président,1998-11-25,NaN,NaN,presidency,11,98.0,1998,2.0,NaN,True,"\"""" ( Sourires. ) \""""",18
78100,122453,Valéry Giscard d’Estaing,1998-11-25,NaN,UDF,mp,11,98.0,1998,2.0,NaN,True,"\"""" ( Sourires. ) \""""",21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80408,126303,le président,1998-11-25,NaN,NaN,presidency,11,99.0,1998,0.0,NaN,True,"\"""" ( Sourires. ) \""""",2329
80460,126390,NaN,1998-11-25,NaN,NaN,gov,11,99.0,1998,0.0,NaN,True,"\"""" ( Applaudissements sur les bancs de le gro...",2381
80470,126408,Nicole Catala,1998-11-25,NaN,RPR,mp,11,99.0,1998,0.0,NaN,True,"\"""" ( Exclamations sur les bancs de le groupe ...",2391
80472,126410,Nicole Catala,1998-11-25,NaN,RPR,mp,11,99.0,1998,0.0,NaN,True,"\"""" ( Mêmes mouvements. ) \""""",2393


In [174]:
party = pd.read_csv('../data/parties/party.csv')
deputes = pd.read_csv('../data/final/politician.csv')

In [175]:
grouped_one_day_speeches = bind_entities(grouped_one_day_speeches, party, deputes)
binding_reports = bind_report(grouped_one_day_speeches)

print('Party binding rows')
display(binding_reports['party_rows'])
print('Unmatched party labels')
display(binding_reports['party_unmatched_labels'])

print('Politician binding rows')
display(binding_reports['politician_rows'])
print('Unmatched speakers')
display(binding_reports['politician_unmatched_speakers'])

grouped_one_day_speeches

Party binding rows


,status,rows
0,matched,1271
1,missing_source,393


Unmatched party labels


,party_source_label,rows


Politician binding rows


,status,rows
0,matched,1271
1,unmatched,318
2,missing_source,75


Unmatched speakers


,speaker,rows
0,le président,310
1,Christine Boutin et M. Philippe de Villiers,2
2,Jacques Myard et M. Philippe de Villiers,2
3,Jacques Myard et M. Michel Bouvard,2
4,Jean-Claude Lefort et M. Julien Dray,1
5,Georges Sarre et M. Hervé de Charette,1


,speech_order,row_type,procedure_type,first_row_order,last_row_order,source_row_orders,source_ids,speaker,role,party,...,party_source_label,party_source_column,party_id,party_acronym,party_name,party_bind_status,mp_id,politician_first_name,politician_last_name,politician_bind_status
0,1,speech,None,1,1,[1],[122427],le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
1,2,procedure,major_topic,2,2,[2],[122428],le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
2,3,speech,None,3,4,"[3, 4]","[122430, 122431]",le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
3,4,procedure,major_topic,5,5,[5],[122432],le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
4,5,speech,None,6,7,"[6, 7]","[122434, 122435]",le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1659,1660,speech,None,2532,2539,"[2532, 2533, 2534, 2535, 2536, 2537, 2538, 2539]","[126636, 126637, 126638, 126639, 126640, 12664...",le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
1660,1661,procedure,major_topic,2540,2540,[2540],[126644],le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
1661,1662,speech,None,2541,2543,"[2541, 2542, 2543]","[126646, 126647, 126648]",le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched
1662,1663,procedure,major_topic,2544,2544,[2544],[126649],le président,presidency,None,...,None,None,<NA>,<NA>,<NA>,missing_source,<NA>,<NA>,<NA>,unmatched


In [194]:
df_final = grouped_one_day_speeches.drop(
    columns=[
        'speaker', 'party', 'parliamentary_group', 'agenda_item',
        'party_source_label', 'party_source_column', 'party_acronym',
        'party_name', 'party_bind_status', 'politician_first_name',
        'politician_last_name', 'politician_bind_status'
    ]
)
for col in ['session', 'major_topic_order']:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce').astype('Int64')

df_final

,speech_order,row_type,procedure_type,first_row_order,last_row_order,source_row_orders,source_ids,role,date,session,major_topic_order,major_topic,subtopic,speech_text,interjections,party_id,mp_id
0,1,speech,None,1,1,[1],[122427],presidency,1998-11-25,98,<NA>,None,None,La séance est ouverte. ( La séance est ouverte...,[],<NA>,<NA>
1,2,procedure,major_topic,2,2,[2],[122428],presidency,1998-11-25,98,1,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,None,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,[],<NA>,<NA>
2,3,speech,None,3,4,"[3, 4]","[122430, 122431]",presidency,1998-11-25,98,1,SOUHAITS DE BIENVENUE AUX PRÉSIDENTS D'ASSEMBL...,None,"Mes chers collègues, sur décision de notre bur...","[( Mmes et MM. les députés, ainsi que Mmes et ...",<NA>,<NA>
3,4,procedure,major_topic,5,5,[5],[122432],presidency,1998-11-25,98,2,QUESTIONS AU GOUVERNEMENT,None,QUESTIONS AU GOUVERNEMENT,[],<NA>,<NA>
4,5,speech,None,6,7,"[6, 7]","[122434, 122435]",presidency,1998-11-25,98,2,QUESTIONS AU GOUVERNEMENT,None,L' ordre de le jour appelle les questions à le...,[],<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1659,1660,speech,None,2532,2539,"[2532, 2533, 2534, 2535, 2536, 2537, 2538, 2539]","[126636, 126637, 126638, 126639, 126640, 12664...",presidency,1998-11-25,99,12,DÉPÔT DE RAPPORTS SUR DES PROPOSITIONS DE RÉSO...,None,"J' ai reçu, le 25 novembre 1998, de M. Yves Du...",[],<NA>,<NA>
1660,1661,procedure,major_topic,2540,2540,[2540],[126644],presidency,1998-11-25,99,13,COMMUNICATION RELATIVE AUX ASSEMBLÉES TERRITOR...,None,COMMUNICATION RELATIVE AUX ASSEMBLÉES TERRITOR...,[],<NA>,<NA>
1661,1662,speech,None,2541,2543,"[2541, 2542, 2543]","[126646, 126647, 126648]",presidency,1998-11-25,99,13,COMMUNICATION RELATIVE AUX ASSEMBLÉES TERRITOR...,None,J' ai reçu de M. le Premier ministre une lettr...,[],<NA>,<NA>
1662,1663,procedure,major_topic,2544,2544,[2544],[126649],presidency,1998-11-25,99,14,ORDRE DU JOUR DES PROCHAINES SÉANCES,None,ORDRE DU JOUR DES PROCHAINES SÉANCES,[],<NA>,<NA>


j'ai deja les mp_id des des president des assemblées depuis la 11eme legislatature j'aimerai : 
quand role president detecter alors on se refere au president associé lors de la date du speech (car plusieurs president assemble national)


| Législature | Président        |                  Parti |             Début |              Fin |    mp_id |
| ----------- | ---------------- | ---------------------- | ----------------- | ---------------- | -------- |
| 11e         | Laurent Fabius   |                     PS |      12 juin 1997 |     27 mars 2000 |   PA1268 |
| 11e         | Raymond Forni    |                     PS |      29 mars 2000 |     18 juin 2002 |   PA1328 |
| 12e         | Jean-Louis Debré |                    UMP |      25 juin 2002 |      4 mars 2007 |   PA1022 |
| 12e         | Patrick Ollier   |                    UMP |       7 mars 2007 |     19 juin 2007 |   PA2295 |
| 13e         | Bernard Accoyer  |                    UMP |      26 juin 2007 |     19 juin 2012 |    PA230 |
| 14e         | Claude Bartolone |                     PS |      26 juin 2012 |     20 juin 2017 |    PA381 |
| 15e         | François de Rugy |                   LREM |      27 juin 2017 | 4 septembre 2018 | PA332747 |
| 15e         | Richard Ferrand  |                   LREM | 12 septembre 2018 |     21 juin 2022 | PA606171 |
| 16e         | Yaël Braun-Pivet |            Renaissance |      28 juin 2022 |      9 juin 2024 | PA721908 |
| 17e         | Yaël Braun-Pivet | Renaissance / Ensemble |   18 juillet 2024 |         en cours | PA721908 |


3/ et si plusieur séance par jour, distinction? // oui et j'utilise session pour faire la distinction car il s'agit du nb de session de l'anner
5/ vérif quality global
6/ fusion parisparl et speech

In [178]:
grouped_one_day_speeches.to_csv('25-11-1998.csv', index=False)

In [189]:
def build_final_structures(df_101, df_103, df_106):
    """
    Prend les 3 DataFrames journaliers et retourne, pour chacun,
    les colonnes :
    major_topic_order, major_topic, subtopic
    """

    inputs = {
        "df_101": df_101,
        "df_103": df_103,
        "df_106": df_106,
    }

    outputs = {}

    for name, one_day_speech in inputs.items():
        debate_structure = extract_debate_structure(one_day_speech)

        major_topics_check = (
            debate_structure[
                debate_structure["structure_level"].isin(["major_topic", "subtopic"])
            ]
            [[
                "structure_level",
                "major_topic_order",
                "major_topic_number",
                "major_topic",
                "subtopic",
                "session",
                "agenda_item",
                "id",
                "structure_text",
            ]]
            .drop_duplicates(
                subset=[
                    "structure_level",
                    "major_topic_order",
                    "major_topic_number",
                    "major_topic",
                    "subtopic",
                ]
            )
            .sort_values(["major_topic_order", "id"])
        )

        final_structure = major_topics_check[
            ["major_topic_order", "major_topic", "subtopic"]
        ].reset_index(drop=True)

        outputs[name] = final_structure

    return outputs

df_101 = df[
    df["date"].eq(pd.Timestamp("1998-11-26"))
    & df["session"].eq(100)
].copy()

df_103 = df[
    df["date"].eq(pd.Timestamp("1998-12-01"))
    & df["session"].eq(103)
].copy()

df_106 = df[
    df["date"].eq(pd.Timestamp("1998-12-02"))
    & df["session"].eq(106)
].copy()

final_structures = build_final_structures(df_101, df_103, df_106)

final_structure_101 = final_structures["df_101"]
final_structure_103 = final_structures["df_103"]
final_structure_106 = final_structures["df_106"]

In [ ]:
# final_structure_101.to_csv('1.csv')
# final_structure_103.to_csv('2.csv')
# final_structure_106.to_csv('3.csv')

#### pipeline over 1998 to 2011 included output expected as csv

In [ ]:
from pathlib import Path

YEARS = range(1998, 2011)
PARISPARL_DIR = Path('../data/speech/1996_2019')
OUTPUT_CSV = Path('../data/final/parisparl_speeches_1998_2010_processed.csv')

FINAL_DROP_COLUMNS = [
    'speaker', 'party', 'parliamentary_group', 'agenda_item',
    'party_source_label', 'party_source_column', 'party_acronym',
    'party_name', 'party_bind_status', 'politician_first_name',
    'politician_last_name', 'politician_bind_status'
]
FINAL_INT_COLUMNS = ['session', 'major_topic_order']


def load_parisparl_years(years=YEARS, data_dir=PARISPARL_DIR) -> pd.DataFrame:
    frames = []
    for year in years:
        path = data_dir / f'parisparl_reden_{year}.csv'
        if not path.exists():
            raise FileNotFoundError(path)

        year_df = pd.read_csv(path)
        year_df['source_year'] = year
        frames.append(year_df)

    output = pd.concat(frames, ignore_index=True)
    output['date'] = pd.to_datetime(output['date'], errors='coerce')
    return output


def process_session_speeches(session_speeches: pd.DataFrame) -> pd.DataFrame:
    session_speeches = session_speeches.copy()
    session_speeches['row_order'] = range(1, len(session_speeches) + 1)

    annotated = annotate_debate_structure(session_speeches)
    return group_consecutive_speeches(
        annotated,
        source_order_col='row_order',
        procedure_col='structure_level',
    )


def build_parisparl_pipeline(years=YEARS) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, pd.DataFrame], pd.DataFrame]:
    raw = load_parisparl_years(years)
    sort_cols = [col for col in ['date', 'session', 'agenda_item', 'id'] if col in raw.columns]
    raw = raw.sort_values(sort_cols, kind='stable').reset_index(drop=True)

    group_cols = [col for col in ['date', 'session'] if col in raw.columns]
    grouped_sessions = []
    for _, session_speeches in raw.groupby(group_cols, dropna=False, sort=True):
        grouped_sessions.append(process_session_speeches(session_speeches))

    grouped = pd.concat(grouped_sessions, ignore_index=True)
    grouped = bind_entities(grouped, party, deputes)
    reports = bind_report(grouped)

    final = grouped.drop(
        columns=[col for col in FINAL_DROP_COLUMNS if col in grouped.columns]
    )
    for col in FINAL_INT_COLUMNS:
        if col in final.columns:
            final[col] = pd.to_numeric(final[col], errors='coerce').astype('Int64')

    return raw, grouped, reports, final


all_speeches_raw, grouped_all_speeches, binding_reports_all, df_final_all = build_parisparl_pipeline()

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_final_all.to_csv(OUTPUT_CSV, index=False)

print(f'Raw rows: {len(all_speeches_raw):,}')
print(f'Final rows: {len(df_final_all):,}')
print(f'CSV written: {OUTPUT_CSV}')

print('Party binding rows')
display(binding_reports_all['party_rows'])
print('Politician binding rows')
display(binding_reports_all['politician_rows'])

df_final_all


/var/folders/kx/6nzn3d5916g_smcrxv435mpc0000gn/T/ipykernel_93788/4182796840.py:54: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  grouped = pd.concat(grouped_sessions, ignore_index=True)


Raw rows: 4,124,555
Final rows: 2,386,200
CSV written: ../data/final/parisparl_speeches_1998_2019_processed.csv
Party binding rows


,status,rows
0,matched,1411646
1,missing_source,724649
2,unmatched,249905


Politician binding rows


,status,rows
0,matched,1632524
1,unmatched,729767
2,missing_source,23909


,speech_order,row_type,procedure_type,first_row_order,last_row_order,source_row_orders,source_ids,role,date,session,major_topic_order,major_topic,subtopic,speech_text,interjections,party_id,mp_id
0,1,speech,None,1,1,[1],[2],presidency,1998-10-01,1,<NA>,None,None,La séance est ouverte. ( La séance est ouverte...,[],<NA>,<NA>
1,2,procedure,major_topic,2,2,[2],[3],presidency,1998-10-01,1,1,OUVERTURE DE LA SESSION,None,OUVERTURE DE LA SESSION,[],<NA>,<NA>
2,3,speech,None,3,3,[3],[5],presidency,1998-10-01,1,1,OUVERTURE DE LA SESSION,None,Conformément à le premier alinéa de l' article...,[],<NA>,<NA>
3,4,speech,None,4,4,[4],[7],mp,1998-10-01,1,1,OUVERTURE DE LA SESSION,None,Très bien!,[],P008,PA1496
4,5,speech,None,5,5,[5],[9],presidency,1998-10-01,1,1,OUVERTURE DE LA SESSION,None,Merci de votre approbation! ( Sourires. ),[],<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2386195,1107,procedure,subtopic,1847,1847,[1847],[6582164],mp,2018-12-20,115,1,MESURES D'URGENCE ÉCONOMIQUESET SOCIALES,VOTE SUR L'ENSEMBLE,VOTE SUR L'ENSEMBLE,[],P020,PA642868
2386196,1108,speech,None,1848,1854,"[1848, 1849, 1850, 1851, 1852, 1853, 1854]","[6582166, 6582167, 6582169, 6582170, 6582171, ...",presidency,2018-12-20,115,1,MESURES D'URGENCE ÉCONOMIQUESET SOCIALES,VOTE SUR L'ENSEMBLE,Je vais maintenant mettre à les voix l' ensemb...,[( Applaudissements sur les bancs des groupes ...,<NA>,<NA>
2386197,1109,speech,None,1855,1858,"[1855, 1856, 1857, 1858]","[6582176, 6582177, 6582178, 6582179]",gov,2018-12-20,115,1,MESURES D'URGENCE ÉCONOMIQUESET SOCIALES,VOTE SUR L'ENSEMBLE,"Monsieur le président, vous qui avez présidé d...",[( Applaudissements sur les bancs des groupes ...,<NA>,PA717169
2386198,1110,procedure,major_topic,1859,1859,[1859],[6582181],presidency,2018-12-20,115,2,ORDRE DU JOUR DE LA PROCHAINE SÉANCE,None,ORDRE DU JOUR DE LA PROCHAINE SÉANCE,[],<NA>,<NA>


In [2]:
df_2011_2026 = pd.read_csv('../data/speech/2011_2026/centralized_speeches.csv')
politician = pd.read_csv('../data/final/politician.csv')
mandate = pd.read_csv('../data/output/mandate.csv')
print(politician.head(1))
print(df_2011_2026.head(1))

/var/folders/kx/6nzn3d5916g_smcrxv435mpc0000gn/T/ipykernel_50151/7964363.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2011_2026 = pd.read_csv('../data/speech/2011_2026/centralized_speeches.csv')


    mp_id     first_name  last_name  birth_date          birth_place  \
0  PA1001  Marc-Philippe  Daubresse  1953-08-01  Lille, Nord, France   

   education_level  school  academic_title  \
0              NaN     NaN             NaN   

                                      profession  languages  seniority  \
0  Ancien directeur d'une société de recrutement        NaN       11.1   

      status  country_id  
0  ASSEMBLEE           3  
           source_files                      id        date  \
0  CRI_20111004_083.xml  ANCR201100000159-00069  03-10-2011   

           text_speech_brut       text_speech_clean topic subtopic1 subtopic2  \
0  . La séance est ouverte.  La séance est ouverte.   NaN       NaN       NaN   

        speaker link_speaker  role gender  speech_interjection  quote  \
0  le président          NaN   NaN      M                  NaN    NaN   

   numSeance  
0        1.0  


In [3]:
df_2011_2026

,source_files,id,date,text_speech_brut,text_speech_clean,topic,subtopic1,subtopic2,speaker,link_speaker,role,gender,speech_interjection,quote,numSeance
0,CRI_20111004_083.xml,ANCR201100000159-00069,03-10-2011,. La séance est ouverte.,La séance est ouverte.,NaN,NaN,NaN,le président,NaN,NaN,M,NaN,NaN,1.0
1,CRI_20111004_083.xml,ANCR201100000159-00073,03-10-2011,Conformément au premier alinéa de l’article 28...,Conformément au premier alinéa de l’article 28...,Ouverture de la session ordinaire,NaN,NaN,le président,NaN,NaN,M,NaN,NaN,1.0
2,CRI_20111004_083.xml,ANCR201100000159-00076,03-10-2011,"M. le ministre de l’intérieur, de l’outre-mer,...","M. le ministre de l’intérieur, de l’outre-mer,...",Cessation de mandat de députés,NaN,NaN,le président,NaN,NaN,M,NaN,NaN,1.0
3,CRI_20111004_083.xml,ANCR201100000159-00077,03-10-2011,"En conséquence, ils cessent d’appartenir à l’A...","En conséquence, ils cessent d’appartenir à l’A...",Cessation de mandat de députés,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,CRI_20111004_083.xml,ANCR201100000159-00080,03-10-2011,"M. le président du groupe socialiste, radical,...","M. le président du groupe socialiste, radical,...",Modification de la composition du bureau de l’...,NaN,NaN,le président,NaN,NaN,M,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3084890,AAA_20260127_013.xml;CRI_20260127_013.xml,w1aab3ab1b1b1i01447,26-01-2026,"Mme Anne Le Hénanff, ministre déléguée. Je rem...","Mme Anne Le Hénanff, ministre déléguée. Je rem...",Protéger les mineurs des risques auxquels les ...,NaN,Vote sur l’ensemble,"Mme Anne Le Hénanff,",NaN,NaN,NaN,NaN,NaN,125.0
3084891,AAA_20260127_013.xml;CRI_20260127_013.xml,w1aab3ab1b1b1i01449,26-01-2026,Comme l’a annoncé le président de la Républiqu...,Comme l’a annoncé le président de la Républiqu...,Protéger les mineurs des risques auxquels les ...,NaN,Vote sur l’ensemble,NaN,NaN,NaN,NaN,NaN,NaN,125.0
3084892,AAA_20260127_013.xml;CRI_20260127_013.xml,w1aab3ab1b1b1i01452,26-01-2026,"Mme la présidente. Prochaine séance, aujourd’h...","Mme la présidente. Prochaine séance, aujourd’h...",Ordre du jour de la prochaine séance,NaN,NaN,Mme la présidente.,NaN,NaN,NaN,NaN,NaN,125.0
3084893,AAA_20260127_013.xml;CRI_20260127_013.xml,w1aab3ab1b1b1i01453,26-01-2026,Questions orales sans débat.,Questions orales sans débat.,Ordre du jour de la prochaine séance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,125.0


In [4]:
from entity_binding import bind_politicians_with_mandates

df_2011_2026 = bind_politicians_with_mandates(
    df_2011_2026,
    politician,
    mandate,
    speaker_col="speaker",
    link_col="link_speaker",  
    date_col="date",
    target_col="id_mp",
)

In [5]:
from entity_binding import (
    merge_missing_speaker_rows,
    missing_speaker_merge_report,
)

df_before_merge = df_2011_2026.copy()

df_2011_2026_merged, missing_speaker_audit = merge_missing_speaker_rows(
    df_2011_2026,
    text_cols=("text_speech_clean", "text_speech_brut"),
    status_col="politician_bind_status",
    missing_status="missing_speaker",
    speaker_col="speaker",
    boundary_cols=("source_files", "date", "numSeance"),
)

report = missing_speaker_merge_report(
    df_before_merge,
    df_2011_2026_merged,
    missing_speaker_audit,
    text_col="text_speech_clean",
    target_col="id_mp",
)

print(report)
print(missing_speaker_audit["status"].value_counts(dropna=False))

rows_before                       3084895
rows_after                        1899687
rows_absorbed                     1185208
missing_speaker_before            1193486
missing_speaker_after                8278
blocked_missing_speaker_rows         8278
blocked_speaker_marker_rows          8273
audit_rows_returned                 10000
audit_rows_sampled                   True
text_chars_before               801788095
text_chars_after                804158237
expected_separator_chars          2370142
text_char_check_pass                 True
pairing_rate_before              0.981301
pairing_rate_after               0.981301
pairing_rate_delta                    0.0
dtype: object
status
blocked_text_starts_with_speaker_marker    8273
merged                                     1722
blocked_boundary_mismatch                     5
Name: count, dtype: int64


In [6]:
df_2011_2026.loc[df_2011_2026["id_mp"].isna(), ["speaker"]]

,speaker
0,le président
1,le président
2,le président
3,NaN
4,le président
...,...
3084889,Mme la présidente.
3084891,NaN
3084892,Mme la présidente.
3084893,NaN


In [7]:
df_2011_2026 = df_2011_2026_merged.reset_index(drop=True)

In [8]:
assert report["text_char_check_pass"] == True
assert report["pairing_rate_after"] >= 0.9813

In [9]:
missing_speaker_audit.loc[
    missing_speaker_audit["status"].ne("merged"),
    ["status", "source_id", "target_id", "target_speaker", "text_speech_clean_preview"]
].head(50)

,status,source_id,target_id,target_speaker,text_speech_clean_preview
70,blocked_text_starts_with_speaker_marker,ANCR201100000159-00217,ANCR201100000159-00199,Jean-Pierre Brard,"Monsieur Plagnol, vous avez dit « pragmatique ..."
153,blocked_text_starts_with_speaker_marker,ANCR201100000159-00400,ANCR201100000159-00399,Valérie Boyer,"Monsieur Gaubert, je comprends très bien vos p..."
508,blocked_text_starts_with_speaker_marker,ANCR201100000160-00808,ANCR201100000160-00806,Daniel Fasquelle,"Monsieur Brottes, je veux vous rassurer : si u..."
647,blocked_text_starts_with_speaker_marker,ANCR201100000161-00363,ANCR201100000161-00362,Patrick Beaudouin,"Monsieur Laffineur, ces derniers mois ont mont..."
1032,blocked_text_starts_with_speaker_marker,ANCR201100000162-00191,ANCR201100000162-00189,Frédéric Lefebvre,"Monsieur Tardy, je suis prêt à réfléchir à d’é..."
1196,blocked_text_starts_with_speaker_marker,ANCR201100000162-00706,ANCR201100000162-00705,Serge Poignant,M. Gaubert a dit qu’il n’y avait pas urgence.
1197,blocked_text_starts_with_speaker_marker,ANCR201100000162-00716,ANCR201100000162-00715,Serge Poignant,M. Lagarde a évoqué un autre identifiant que l...
1320,blocked_text_starts_with_speaker_marker,ANCR201100000163-00313,ANCR201100000163-00309,Valérie Rosso-Debord,M. François Hollande a indiqué que la scolaris...
1672,blocked_text_starts_with_speaker_marker,ANCR201100000164-00062,ANCR201100000164-00057,Michel Mercier,"Monsieur Ciotti, vous insistez clairement et à..."
1716,blocked_text_starts_with_speaker_marker,ANCR201100000164-00150,ANCR201100000164-00149,Jean-Jacques Urvoas,"M. Michel Mercier, garde des sceaux. Ils arriv..."


In [10]:
df_2011_2026.columns

Index(['source_files', 'id', 'date', 'text_speech_brut', 'text_speech_clean',
       'topic', 'subtopic1', 'subtopic2', 'speaker', 'link_speaker', 'role',
       'gender', 'speech_interjection', 'quote', 'numSeance', 'id_mp',
       'politician_first_name', 'politician_last_name', 'politician_status',
       'politician_name_score', 'politician_bind_status',
       'politician_bind_method', 'politician_mandate_start',
       'politician_mandate_end', 'politician_date_source', 'politician_term',
       'speaker_link_mp_ids', 'speaker_link_mp_id_exists',
       'merged_missing_speaker_rows', 'merged_missing_speaker_ids'],
      dtype='object')

In [11]:
df_2011_2026.drop(columns=['id', 'text_speech_brut', 'gender'], inplace=True, errors='ignore')
df_2011_2026

a = df_2011_2026[["text_speech_clean", "speaker"]]
a.tail(10)

,text_speech_clean,speaker
1899677,"M. Laurent Marcangeli. Cela étant dit, le text...",M. Laurent Marcangeli.
1899678,Mme la présidente. La parole est à Mme Béatric...,Mme la présidente.
1899679,Mme Béatrice Roullaud (RN). Mon collègue Thier...,Mme Béatrice Roullaud (RN).
1899680,M. Arnaud Saint-Martin. Ce n’est pas ce qu’on ...,M. Arnaud Saint-Martin.
1899681,Mme Béatrice Roullaud. Au Rassemblement nation...,Mme Béatrice Roullaud.
1899682,Mme la présidente. Je mets aux voix l’ensemble...,Mme la présidente.
1899683,Mme la présidente. Voici le résultat du scrutin :,Mme la présidente.
1899684,Mme la présidente. La parole est à Mme la mini...,Mme la présidente.
1899685,"Mme Anne Le Hénanff, ministre déléguée. Je rem...","Mme Anne Le Hénanff,"
1899686,"Mme la présidente. Prochaine séance, aujourd’h...",Mme la présidente.


In [12]:
# test = pd.read_csv(
#     "../data/speech/2011_2026/centralized_speeches.csv",
#     usecols=["text_speech_clean", "speaker"]
# )

# test.tail(10)

In [13]:
from entity_binding import normalize_person_key

# Hors présidents
president_keys = {
    "president",
    "le president",
    "la presidente",
    "monsieur le president",
    "madame la presidente",
}

speaker_key = df_2011_2026["speaker"].map(normalize_person_key)

mask_non_president = ~speaker_key.isin(president_keys)
mask_non_president &= speaker_key.ne("")

df_np = df_2011_2026.loc[mask_non_president].copy()

total_rows = len(df_np)
matched_rows = df_np["id_mp"].notna().sum()
unmatched_rows = total_rows - matched_rows
ratio_rows = matched_rows / total_rows if total_rows else 0

print("=== Statistiques de pairage hors présidents ===")
print(f"Lignes hors président : {total_rows:,}")
print(f"Lignes pairées : {matched_rows:,}")
print(f"Lignes non pairées : {unmatched_rows:,}")
print(f"Taux de pairage lignes : {ratio_rows:.2%}")

=== Statistiques de pairage hors présidents ===
Lignes hors président : 1,275,663
Lignes pairées : 1,251,810
Lignes non pairées : 23,853
Taux de pairage lignes : 98.13%


In [14]:
df_np.loc[df_np["id_mp"].isna(), "speaker"].value_counts().head(30)

speaker
Un député du groupe LR.                          1133
Plusieurs députés du groupe LR.                  1042
Un député du groupe RN.                           895
Plusieurs députés du groupe UMP.                  792
Plusieurs députés du groupe UMP .                 679
Un député du groupe LaREM.                        511
Plusieurs députés du groupe SRC.                  503
Plusieurs députés du groupe RN.                   499
Plusieurs députés du groupe LaREM.                437
Un député du groupe LFI-NUPES.                    381
Plusieurs députés du groupe Les Républicains.     343
Plusieurs députés du groupe LFI-NUPES.            301
Un député du groupe LFI-NFP.                      241
Un député du groupe Les Républicains.             227
Plusieurs députés du groupe SRC .                 220
Un député du groupe RE.                           219
Un député du groupe UMP .                         180
Plusieurs députés du groupe RE.                   179
Plusieurs députés du

2/ bind les speakers 
2.5/ suppresion colonnes 
3/ merge 1996_2010 a 2011_2026 
4/ si speaker == le president ou la presidente avoir un id_mp associé dans dans politicians.csv + avoir un mandats associé de president de l'assembleé national



=== Statistiques de pairage hors présidents ===
Lignes hors président : 1,275,663
Lignes pairées : 1,251,810
Lignes non pairées : 23,853
Taux de pairage lignes : 98.13%

A) Structure Kamil (Parisparl + offical only speech): 
*speech_id
speech_text
speech_date
speaker_id
country_id
term
speech_interjection

B) Full Parisparl

C) Full official  

D) Policorp : Parisparl + offical structure + speech ask Nina
    -  garder num de lignes 
    - garder les procedures et speech etc 

E) Deputes and their mandates

F) political party


In [27]:
import re
import unicodedata

import pandas as pd

FINAL_COLUMNS = [
    "speech_id",
    "speech_text",
    "speech_interjection",
    "speech_date",
    "speaker_id",
    "country_id",
    "term",
]
COUNTRY_ID = 3


def require_columns(frame: pd.DataFrame, columns: list[str], frame_name: str) -> None:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{frame_name}: colonnes manquantes: {missing}")


def has_value(series: pd.Series) -> pd.Series:
    return series.notna() & series.astype("string").str.strip().fillna("").ne("")


def strip_accents(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value)
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_label(value: object) -> str:
    if pd.isna(value):
        return ""
    text = strip_accents(str(value).strip().lower())
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def is_assembly_president_label(value: object) -> bool:
    label = normalize_label(value)
    if label == "presidency":
        return True
    if any(word in label for word in ("commission", "groupe", "delegation", "rapporteur")):
        return False

    president_prefixes = (
        "m le president",
        "m m le president",
        "m la presidente",
        "mme la presidente",
        "mme le presidente",
        "monsieur le president",
        "madame la presidente",
        "le president",
        "la presidente",
        "la présidente",
        "le président",
    )
    return any(label == prefix or label.startswith(f"{prefix} ") for prefix in president_prefixes)


def parse_speech_dates(series: pd.Series) -> pd.Series:
    raw = series.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    iso_mask = raw.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    dmy_mask = raw.str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)

    parsed.loc[iso_mask] = pd.to_datetime(raw.loc[iso_mask], format="%Y-%m-%d", errors="coerce")
    parsed.loc[dmy_mask] = pd.to_datetime(raw.loc[dmy_mask], format="%d-%m-%Y", errors="coerce")

    fallback_mask = parsed.isna() & raw.notna() & raw.ne("")
    parsed.loc[fallback_mask] = pd.to_datetime(raw.loc[fallback_mask], errors="coerce", dayfirst=True)
    return parsed


def parse_dates_from_text(series: pd.Series) -> pd.Series:
    raw = series.astype("string")
    extracted = raw.str.extract(r"(?<!\d)((?:19|20)\d{2})(\d{2})(\d{2})(?!\d)")
    date_text = extracted[0] + "-" + extracted[1] + "-" + extracted[2]
    return pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")


def add_term_from_date(frame: pd.DataFrame, date_col: str = "_speech_date_parsed") -> pd.DataFrame:
    legislature_ranges = pd.DataFrame(
        [
            (11, "1997-06-12", "2002-06-18"),
            (12, "2002-06-19", "2007-06-19"),
            (13, "2007-06-20", "2012-06-19"),
            (14, "2012-06-20", "2017-06-20"),
            (15, "2017-06-21", "2022-06-21"),
            (16, "2022-06-22", "2024-06-09"),
            (17, "2024-07-01", "2100-12-31"),
        ],
        columns=["term", "start_date", "end_date"],
    )
    legislature_ranges["start_date"] = pd.to_datetime(legislature_ranges["start_date"])
    legislature_ranges["end_date"] = pd.to_datetime(legislature_ranges["end_date"])

    output = frame.copy()
    output["term"] = pd.Series(pd.NA, index=output.index, dtype="Int64")
    for _, legislature in legislature_ranges.iterrows():
        in_legislature = output[date_col].between(
            legislature["start_date"],
            legislature["end_date"],
            inclusive="both",
        )
        output.loc[in_legislature, "term"] = int(legislature["term"])
    return output


def bind_presidency_ids(frame: pd.DataFrame, presidency: pd.DataFrame) -> pd.DataFrame:
    require_columns(presidency, ["pr_id", "start_date", "end_date"], "presidency")

    presidents = presidency.copy()
    presidents["_start_date"] = parse_speech_dates(presidents["start_date"])
    presidents["_end_date"] = parse_speech_dates(presidents["end_date"]).fillna(pd.Timestamp("2100-12-31"))

    output = frame.copy()
    is_presidency = output["role"].map(is_assembly_president_label)
    missing_speaker_id = ~has_value(output["speaker_id"])

    for _, president in presidents.sort_values("_start_date").iterrows():
        in_presidency = output["_speech_date_parsed"].between(
            president["_start_date"],
            president["_end_date"],
            inclusive="both",
        )
        fill_mask = is_presidency & missing_speaker_id & in_presidency
        output.loc[fill_mask, "speaker_id"] = president["pr_id"]
        missing_speaker_id = ~has_value(output["speaker_id"])

    return output


# 1) df_2011_2026 -> df_2011_2026_final
required_2011_columns = ["date", "text_speech_clean", "id_mp", "speaker", "speech_interjection"]
require_columns(df_2011_2026, required_2011_columns, "df_2011_2026")
optional_2011_columns = [column for column in ["source_files"] if column in df_2011_2026.columns]

_df_2011_2026_work = df_2011_2026[required_2011_columns + optional_2011_columns].copy()
_df_2011_2026_work["_source_dataset"] = 2
_df_2011_2026_work["_source_order"] = range(len(_df_2011_2026_work))
_df_2011_2026_work["_speech_date_parsed"] = parse_speech_dates(_df_2011_2026_work["date"])
if "source_files" in _df_2011_2026_work.columns:
    _df_2011_2026_work["_speech_date_parsed"] = _df_2011_2026_work["_speech_date_parsed"].fillna(
        parse_dates_from_text(_df_2011_2026_work["source_files"])
    )

_df_2011_2026_work = _df_2011_2026_work.rename(
    columns={
        "text_speech_clean": "speech_text",
        "date": "speech_date",
        "id_mp": "speaker_id",
        "speaker": "role",
    }
)

mask_2011_has_speaker = has_value(_df_2011_2026_work["role"])
mask_2011_has_speaker_id = has_value(_df_2011_2026_work["speaker_id"])
mask_2011_president = _df_2011_2026_work["role"].map(is_assembly_president_label)
mask_2011_has_date = _df_2011_2026_work["_speech_date_parsed"].notna()

# On garde les presidents sans id_mp pour pouvoir les relier ensuite a presidency.pr_id.
df_2011_2026_final = _df_2011_2026_work.loc[
    mask_2011_has_date & mask_2011_has_speaker & (mask_2011_has_speaker_id | mask_2011_president)
].copy()
df_2011_2026_final["speech_date"] = df_2011_2026_final["_speech_date_parsed"].dt.strftime("%d-%m-%Y")


# 2) df_1998_2010 -> df_1998_2010_final
df_1998_2010 = pd.read_csv("../data/final/parisparl_speeches_1998_2010_processed.csv")
required_1998_columns = ["speech_text", "date", "mp_id", "role", "interjections", "row_type"]
require_columns(df_1998_2010, required_1998_columns, "df_1998_2010")

_df_1998_2010_work = df_1998_2010.copy()
_df_1998_2010_work["_source_dataset"] = 1
_df_1998_2010_work["_source_order"] = range(len(_df_1998_2010_work))
_df_1998_2010_work["_speech_date_parsed"] = parse_speech_dates(_df_1998_2010_work["date"])

# Dans ce CSV, row_type porte la valeur "speech"; speech_order est un rang numerique.
mask_1998_2010_dates = _df_1998_2010_work["_speech_date_parsed"].dt.year.between(1998, 2010)
mask_1998_2010_speech = _df_1998_2010_work["row_type"].astype("string").str.lower().eq("speech")

_df_1998_2010_work = _df_1998_2010_work.loc[
    mask_1998_2010_dates & mask_1998_2010_speech,
    [
        "speech_text",
        "date",
        "mp_id",
        "role",
        "interjections",
        "_source_dataset",
        "_source_order",
        "_speech_date_parsed",
    ],
].copy()

df_1998_2010_final = _df_1998_2010_work.rename(
    columns={
        "date": "speech_date",
        "mp_id": "speaker_id",
        "interjections": "speech_interjection",
    }
)
df_1998_2010_final["speech_date"] = df_1998_2010_final["_speech_date_parsed"].dt.strftime("%d-%m-%Y")


# 3) Fusion 1998-2026 + president_id + colonnes finales
presidency = pd.read_csv("../data/final/presidency.csv")

df_1998_2026_work = pd.concat(
    [df_1998_2010_final, df_2011_2026_final],
    ignore_index=True,
    sort=False,
)
df_1998_2026_work = bind_presidency_ids(df_1998_2026_work, presidency)
df_1998_2026_work = add_term_from_date(df_1998_2026_work)

missing_speaker_id_after_presidency = (~has_value(df_1998_2026_work["speaker_id"])).sum()
df_1998_2026_work = df_1998_2026_work.loc[has_value(df_1998_2026_work["speaker_id"])].copy()

df_1998_2026_work = df_1998_2026_work.sort_values(
    ["_speech_date_parsed", "_source_dataset", "_source_order"],
    kind="stable",
).reset_index(drop=True)

df_1998_2026_work = df_1998_2026_work.dropna(subset=["speech_text"]).reset_index(drop=True)
df_1998_2026_work["speech_id"] = range(1, len(df_1998_2026_work) + 1)
df_1998_2026_work["country_id"] = COUNTRY_ID

df_1998_2026 = df_1998_2026_work[FINAL_COLUMNS].copy()

quality_report = pd.Series(
    {
        "df_1998_2010_final_rows": len(df_1998_2010_final),
        "df_2011_2026_final_rows": len(df_2011_2026_final),
        "df_1998_2026_rows": len(df_1998_2026),
        "dropped_missing_speaker_id_after_presidency": int(missing_speaker_id_after_presidency),
        "missing_speech_date": int(df_1998_2026["speech_date"].isna().sum()),
        "missing_speaker_id": int((~has_value(df_1998_2026["speaker_id"])).sum()),
        "missing_term": int(df_1998_2026["term"].isna().sum()),
    }
)
print(quality_report.to_string())

assert list(df_1998_2026.columns) == FINAL_COLUMNS
assert df_1998_2026["speech_id"].is_unique
assert df_1998_2026["speech_id"].is_monotonic_increasing
assert df_1998_2026["speech_date"].notna().all()
assert has_value(df_1998_2026["speaker_id"]).all()
assert df_1998_2026["country_id"].eq(COUNTRY_ID).all()
assert df_1998_2026["term"].notna().all()

df_1998_2026

df_1998_2010_final_rows                        1396350
df_2011_2026_final_rows                        1866073
df_1998_2026_rows                              3204957
dropped_missing_speaker_id_after_presidency      57464
missing_speech_date                                  0
missing_speaker_id                                   0
missing_term                                         0


,speech_id,speech_text,speech_interjection,speech_date,speaker_id,country_id,term
0,1,La séance est ouverte. ( La séance est ouverte...,[],01-10-1998,PRNA11,3,11
1,2,Conformément à le premier alinéa de l' article...,[],01-10-1998,PRNA11,3,11
2,3,Très bien!,[],01-10-1998,PA1496,3,11
3,4,Merci de votre approbation! ( Sourires. ),[],01-10-1998,PRNA11,3,11
4,5,"à les termes de l' article 10 de le règlement,...","['( Applaudissements sur quelques bancs )', '(...",01-10-1998,PRNA11,3,11
...,...,...,...,...,...,...,...
3204952,3204953,Mme la présidente. Je mets aux voix l’ensemble...,NaN,26-01-2026,PRNA17,3,17
3204953,3204954,Mme la présidente. Voici le résultat du scrutin :,NaN,26-01-2026,PRNA17,3,17
3204954,3204955,Mme la présidente. La parole est à Mme la mini...,NaN,26-01-2026,PRNA17,3,17
3204955,3204956,"Mme Anne Le Hénanff, ministre déléguée. Je rem...",NaN,26-01-2026,PA794426,3,17


In [ ]:
# df_1998_2026.to_csv("../data/final/speeches_1998_2026.csv", index=False)

In [28]:
df_1998_2026.isna().sum()

speech_id                    0
speech_text                  0
speech_interjection    1866072
speech_date                  0
speaker_id                   0
country_id                   0
term                         0
dtype: int64

In [26]:
lignes_manquantes = df_1998_2026[df_1998_2026["speech_text"].isna()]
lignes_manquantes

,speech_id,speech_text,speech_interjection,speech_date,speaker_id,country_id,term
959670,959671,NaN,[],26-09-2007,PA1206,3,13
1343911,1343912,NaN,NaN,17-10-2011,PA1252,3,13


1) voir ce qui a été supprimer : dropped_missing_speaker_id_after_presidency
2) fusion entre jeu de donnée dila + parisparl 
3) update la bdd et push code sur gitlab
4) realisation de la doc + latex rapport